# Deep clustering baselines
DEC, DCN, VaDE and the CNN pseudo-label experiment come from `mnist_baseline.ipynb`. The joint k-means experiment also uses `mnist_vae_kmeans (1).ipynb`.

IDEC and the ADMM experiments were reconstructed for the thesis comparison in the previous GitHub version. The CNN pseudo-label method and ADMM DC+k-means are different algorithms, so they keep separate names here. The short settings at the end are for trying the methods; they are not the thesis experiment budget.

In [ ]:
%run ./functions.ipynb

import math
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from torch.utils.data import WeightedRandomSampler

## A few shared steps
The sample indices keep DEC targets and ADMM variables aligned even when batches are shuffled. Training below never uses the true labels.

In [ ]:
def indexed_batches(X, batch_size=256, shuffle=True):
    return DataLoader(TensorDataset(X, torch.arange(len(X))),
                      batch_size=batch_size, shuffle=shuffle)


def gradient_step(loss, optimizer, parameters):
    if not torch.isfinite(loss):
        raise FloatingPointError("The training loss became non-finite.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(parameters, 10.)
    optimizer.step()


def prepare_ae(X, dimensions, pretrain_epochs, batch_size, lr, seed):
    dimensions = dimensions or [X.shape[1], 500, 500, 2000, 10]
    model = AutoEncoder(dimensions, dropout=0.).to(device)
    pre_train(X, model, epochs=pretrain_epochs, batch_size=batch_size,
              lr=lr, loss='mse', seed=seed)
    return model, vae_encode(X, model, batch_size)

## DEC and IDEC
DEC sharpens the Student-t assignment distribution and minimizes $KL(P\|Q)$. IDEC adds reconstruction to the same training loop. Targets are recomputed every `update_interval` epochs; the final predictions are always recomputed after training. SGD follows the original DEC notebook; `optimizer_name='adam'` retains the later GitHub variant.

In [ ]:
class DEC(nn.Module):
    def __init__(self, autoencoder, centers, alpha=1.):
        super().__init__()
        self.autoencoder = autoencoder
        self.cluster_centers = nn.Parameter(torch.as_tensor(centers, dtype=torch.float32))
        self.alpha = alpha

    def encode(self, x):
        return self.autoencoder.encode(x)

    def compute_q(self, z):
        distance = torch.cdist(z, self.cluster_centers).square()
        q = (1 + distance / self.alpha).pow(-(self.alpha + 1) / 2)
        return q / q.sum(1, keepdim=True).clamp_min(1e-12)

    @staticmethod
    def target_distribution(q):
        weight = q.square() / q.sum(0).clamp_min(1e-12)
        return weight / weight.sum(1, keepdim=True).clamp_min(1e-12)

    def forward(self, x):
        z = self.encode(x)
        return self.compute_q(z), z


@torch.no_grad()
def dec_probabilities(X, model, batch_size=256):
    model.eval()
    return torch.cat([model(x.to(device))[0].cpu()
                      for x, _ in indexed_batches(X, batch_size, False)])


def train_dec(X, n_clusters=10, dimensions=None, pretrain_epochs=10,
              epochs=30, batch_size=256, lr=1e-3, seed=42,
              idec=False, reconstruction_weight=1., update_interval=1, tol=0.,
              optimizer_name='sgd', momentum=.9):
    if update_interval < 1:
        raise ValueError("update_interval must be positive")
    seed_everything(seed)
    ae, Z = prepare_ae(X, dimensions, pretrain_epochs, batch_size, lr, seed)
    centers = KMeans(n_clusters, n_init=10, random_state=seed).fit(Z).cluster_centers_
    model = DEC(ae, centers).to(device)
    if optimizer_name == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    elif optimizer_name == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("optimizer_name must be sgd or adam")
    history = {'loss': [], 'kl_loss': [], 'recon_loss': [], 'changed_labels': []}
    previous = None
    changed = 1.
    for epoch in range(epochs):
        if epoch % update_interval == 0:
            q_all = dec_probabilities(X, model, batch_size)
            p_all = model.target_distribution(q_all).detach()
            current = q_all.argmax(1)
            changed = 1. if previous is None else (current != previous).float().mean().item()
            previous = current
            if epoch > 0 and changed < tol:
                break
        model.train()
        total = np.zeros(3)
        for x, idx in indexed_batches(X, batch_size):
            x = x.to(device)
            q, z = model(x)
            kl = F.kl_div(q.clamp_min(1e-12).log(), p_all[idx].to(device),
                          reduction='batchmean')
            rec = F.mse_loss(ae.decoder(z), x) if idec else x.new_tensor(0.)
            loss = kl + reconstruction_weight * rec
            gradient_step(loss, optimizer, model.parameters())
            total += len(x) * np.array([loss.item(), kl.item(), rec.item()])
        for name, value in zip(('loss', 'kl_loss', 'recon_loss'), total / len(X)):
            history[name].append(value)
        history['changed_labels'].append(changed)
    q = dec_probabilities(X, model, batch_size)
    return {'model': model, 'Z': vae_encode(X, model, batch_size),
            'pred': q.argmax(1).numpy(), 'history': history}

## DCN
This keeps the original BatchNorm autoencoder and cumulative centroid updates (initial count 100). Reconstruction is elementwise MSE, as in the original DCN cells. The centroid term sums latent dimensions and averages samples.

In [ ]:
class SmallBatchNorm1d(nn.BatchNorm1d):
    def forward(self, x):
        # Use running statistics for a final batch containing one sample.
        if self.training and len(x) == 1:
            return F.batch_norm(x, self.running_mean, self.running_var,
                                self.weight, self.bias, False, self.momentum, self.eps)
        return super().forward(x)


class DCNAutoEncoder(AutoEncoder):
    def __init__(self, dimensions):
        nn.Module.__init__(self)
        self.dimensions = list(dimensions)
        self.hidden_dimension = dimensions[-1]
        enc, dec = [], []
        for a, b in zip(dimensions[:-2], dimensions[1:-1]):
            enc += [nn.Linear(a, b), nn.ReLU(), SmallBatchNorm1d(b)]
        enc += [nn.Linear(dimensions[-2], dimensions[-1])]
        reverse = dimensions[::-1]
        for a, b in zip(reverse[:-2], reverse[1:-1]):
            dec += [nn.Linear(a, b), nn.ReLU(), SmallBatchNorm1d(b)]
        dec += [nn.Linear(dimensions[1], dimensions[0])]
        self.encoder = nn.Sequential(*enc)
        self.decoder = nn.Sequential(*dec)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encode(x)
        return self.decoder(z), z


class BatchKMeans:
    def __init__(self, centers, initial_count=100.):
        self.C = torch.as_tensor(centers, dtype=torch.float32, device=device)
        self.count = torch.full((len(centers),), float(initial_count), device=device)

    @torch.no_grad()
    def assign(self, Z):
        return torch.cdist(Z, self.C).argmin(1)

    @torch.no_grad()
    def update_centers(self, Z, labels):
        # The batch sum gives the same cumulative mean as the original inner loop.
        for k in range(len(self.C)):
            selected = Z[labels == k]
            if len(selected):
                self.C[k] = (self.count[k] * self.C[k] + selected.sum(0)) / (self.count[k] + len(selected))
                self.count[k] += len(selected)


def train_dcn(X, n_clusters=10, dimensions=None, pretrain_epochs=10,
              epochs=30, batch_size=256, lr=1e-3, seed=42,
              cluster_weight=1., reconstruction_weight=1., weight_decay=5e-4):
    seed_everything(seed)
    dimensions = dimensions or [X.shape[1], 500, 500, 2000, 10]
    model = DCNAutoEncoder(dimensions).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for epoch in range(pretrain_epochs):
        model.train()
        for x, _ in indexed_batches(X, batch_size):
            x = x.to(device)
            loss = F.mse_loss(model(x)[0], x)
            gradient_step(loss, optimizer, model.parameters())
    Z = vae_encode(X, model, batch_size)
    centers = KMeans(n_clusters, n_init=10, random_state=seed).fit(Z).cluster_centers_
    km = BatchKMeans(centers)
    history = {'loss': [], 'recon_loss': [], 'cluster_loss': []}
    for epoch in range(epochs):
        model.train()
        total = np.zeros(3)
        for x, _ in indexed_batches(X, batch_size):
            x = x.to(device)
            recon, z = model(x)
            label = km.assign(z.detach())
            km.update_centers(z.detach(), label)
            rec = F.mse_loss(recon, x)
            cluster = .5 * (z - km.C[label]).square().sum(1).mean()
            loss = reconstruction_weight * rec + cluster_weight * cluster
            gradient_step(loss, optimizer, model.parameters())
            total += len(x) * np.array([loss.item(), rec.item(), cluster.item()])
        for name, value in zip(history, total / len(X)):
            history[name].append(value)
    Z = vae_encode(X, model, batch_size)
    pred = km.assign(torch.as_tensor(Z, device=device)).cpu().numpy()
    model.register_buffer('cluster_centers', km.C.clone())
    return {'model': model, 'Z': Z, 'pred': pred, 'history': history,
            'centers': km.C.cpu().numpy(), 'counts': km.count.cpu().numpy()}

## DC+k-means / DC+GMM with ADMM
These are reconstructed alternating baselines, not recovered original training scripts. With scaled dual variable $U$, the network penalty is $\|Z-Y+U\|^2$; the auxiliary update uses $Z+U$, then $U\leftarrow U+Z-Y$. The GMM variant fixes responsibilities during each quadratic auxiliary update. It is an experimental extension, not a claim of reproducing a particular published DC+GMM implementation.

In [ ]:
def train_dc_admm(X, n_clusters=10, dimensions=None, pretrain_epochs=10,
                  epochs=30, batch_size=256, lr=1e-3, seed=42,
                  clustering='kmeans', rho=1., cluster_weight=1., gmm_max_iter=100):
    if clustering not in ('kmeans', 'gmm') or rho <= 0 or cluster_weight < 0:
        raise ValueError("Use kmeans/gmm, rho > 0 and cluster_weight >= 0")
    seed_everything(seed)
    model, Z = prepare_ae(X, dimensions, pretrain_epochs, batch_size, lr, seed)
    Y, dual = Z.copy(), np.zeros_like(Z)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': [], 'primal_residual': []}
    if clustering == 'kmeans':
        clusterer = KMeans(n_clusters, n_init=10, random_state=seed)
    else:
        clusterer = GaussianMixture(n_clusters, covariance_type='diag',
                                    reg_covar=1e-3, random_state=seed, max_iter=gmm_max_iter)
    for epoch in range(epochs):
        clusterer.fit(Y)
        model.train()
        total = 0.
        for x, idx in indexed_batches(X, batch_size):
            x = x.to(device)
            recon, z = model(x)
            target = torch.as_tensor(Y[idx.numpy()] - dual[idx.numpy()], device=device)
            rec = (recon - x).square().sum(1).mean()
            loss = rec + .5 * rho * (z - target).square().sum(1).mean()
            gradient_step(loss, optimizer, model.parameters())
            total += len(x) * loss.item()
        Z = vae_encode(X, model, batch_size)
        if clustering == 'kmeans':
            labels = clusterer.predict(Y)
            Y = (cluster_weight * clusterer.cluster_centers_[labels] + rho * (Z + dual)) / (cluster_weight + rho)
        else:
            resp = clusterer.predict_proba(Y)
            precision = 1 / clusterer.covariances_
            numerator = cluster_weight * (resp @ (clusterer.means_ * precision)) + rho * (Z + dual)
            Y = numerator / (cluster_weight * (resp @ precision) + rho)
        Y = Y.astype(np.float32)
        dual += Z - Y
        history['loss'].append(total / len(X))
        history['primal_residual'].append(float(np.linalg.norm(Z - Y) / np.sqrt(len(Y))))
    clusterer.fit(Y)
    return {'model': model, 'Z': Z, 'Y': Y, 'dual': dual, 'clusterer': clusterer,
            'pred': clusterer.predict(Y), 'history': history}

## VaDE
The mixture probabilities use a softmax, so they remain positive and sum to one. The loss uses the expected Gaussian KL plus $KL(q(c\mid x)\|p(c))$; this corrects the categorical entropy sign in the early notebook. Responsibilities use the posterior mean and variance rather than detaching the mean.

In [ ]:
class VaDE(nn.Module):
    def __init__(self, vae, n_centroid, latent_dim):
        super().__init__()
        self.vae = vae
        self.pi_logits = nn.Parameter(torch.zeros(n_centroid))
        self.u_mu = nn.Parameter(torch.zeros(n_centroid, latent_dim))
        self.log_sigma2 = nn.Parameter(torch.zeros(n_centroid, latent_dim))

    def encode(self, x):
        return self.vae.encode(x)

    def responsibilities(self, mu, logvar):
        prior_logvar = self.log_sigma2.clamp(-10, 10)
        expected_log_p = -.5 * (math.log(2 * math.pi) + prior_logvar[None] +
            (logvar.exp()[:, None] + (mu[:, None] - self.u_mu[None]).square()) /
            prior_logvar.exp()[None]).sum(2)
        return torch.softmax(expected_log_p + F.log_softmax(self.pi_logits, dim=0), dim=1)

    def elbo_loss(self, x, reconstruction='bce'):
        recon, mu, logvar = self.vae(x)
        gamma = self.responsibilities(mu, logvar)
        prior_logvar = self.log_sigma2.clamp(-10, 10)
        gaussian_kl = .5 * (prior_logvar[None] - logvar[:, None] - 1 +
            (logvar.exp()[:, None] + (mu[:, None] - self.u_mu[None]).square()) /
            prior_logvar.exp()[None]).sum(2)
        categorical_kl = (gamma * (gamma.clamp_min(1e-12).log() -
                                      F.log_softmax(self.pi_logits, dim=0))).sum(1)
        rec, _ = Loss_Function(recon, x, mu, logvar, kind=reconstruction)
        return rec + (gamma * gaussian_kl).sum(1).mean() + categorical_kl.mean()


@torch.no_grad()
def predict_vade(X, model, batch_size=256):
    model.eval()
    labels = []
    for x, _ in indexed_batches(X, batch_size, False):
        _, mu, logvar = model.vae(x.to(device))
        labels.append(model.responsibilities(mu, logvar).argmax(1).cpu())
    return torch.cat(labels).numpy()


def train_vade(X, n_clusters=10, dimensions=None, pretrain_epochs=10,
               epochs=30, batch_size=256, lr=1e-3, seed=42, reconstruction='bce'):
    seed_everything(seed)
    dimensions = dimensions or [X.shape[1], 500, 500, 2000, 10]
    vae = VariationalAutoEncoder(dimensions, dropout=0.).to(device)
    pre_train(X, vae, epochs=pretrain_epochs, batch_size=batch_size,
              lr=lr, loss=reconstruction, seed=seed)
    Z = vae_encode(X, vae, batch_size)
    gmm = GaussianMixture(n_clusters, covariance_type='diag', reg_covar=1e-2,
                          random_state=seed).fit(Z)
    model = VaDE(vae, n_clusters, dimensions[-1]).to(device)
    with torch.no_grad():
        model.u_mu.copy_(torch.as_tensor(gmm.means_, device=device))
        model.log_sigma2.copy_(torch.as_tensor(np.log(gmm.covariances_), device=device))
        model.pi_logits.copy_(torch.as_tensor(np.log(gmm.weights_), device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': []}
    for epoch in range(epochs):
        model.train()
        total = 0.
        for x, _ in indexed_batches(X, batch_size):
            x = x.to(device)
            loss = model.elbo_loss(x, reconstruction)
            gradient_step(loss, optimizer, model.parameters())
            total += len(x) * loss.item()
        history['loss'].append(total / len(X))
    return {'model': model, 'Z': vae_encode(X, model, batch_size),
            'pred': predict_vade(X, model, batch_size), 'history': history}

## CNN + pseudo-labels (the original `DCKMeans` experiment)
Extract CNN features, optionally whiten with PCA, normalize, cluster, and train against the cluster labels. `reassign` controls how often labels and the classification head are reset. Sobel filtering and late BatchNorm freezing are retained. The learning-rate decay follows the total epoch count, rather than restarting at every reassignment.

Set `pca_dim=None` to skip PCA. The original 128-dimensional features with `pca_dim=256` also skipped PCA. `balance_clusters=True` adds inverse-frequency sampling; the original grouping code did not actually resample.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, side=28, sobel=True):
        super().__init__()
        self.side, self.sobel = side, sobel
        sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]]).reshape(1, 1, 3, 3)
        self.register_buffer('sobel_kernels', torch.cat([sx, sx.transpose(2, 3)], dim=0))
        self.features = nn.Sequential(
            nn.Conv2d(2 if sobel else 1, 64, 5, padding=2), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(3, 2, 1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(3, 2, 1),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        small = math.ceil(math.ceil(math.ceil(side / 2) / 2) / 2)
        self.classifier = nn.Sequential(
            nn.Linear(256 * small * small, 1024), SmallBatchNorm1d(1024),
            nn.ReLU(), nn.Dropout(.5), nn.Linear(1024, 256),
            SmallBatchNorm1d(256), nn.ReLU(), nn.Dropout(.5))
        self.projection = nn.Linear(256, 128)

    def encode(self, x):
        x = x.reshape(-1, 1, self.side, self.side)
        if self.sobel:
            x = F.conv2d(x, self.sobel_kernels, padding=1)
        z = self.projection(self.classifier(self.features(x).flatten(1)))
        return F.normalize(z, dim=1)

    def forward(self, x):
        return self.encode(x)


def cluster_cnn_features(Z, n_clusters, seed=42, pca_dim=64):
    pca = None
    if pca_dim is not None and pca_dim < Z.shape[1]:
        n_components = min(pca_dim, len(Z) - 1, Z.shape[1])
        if n_components < 1:
            raise ValueError("PCA needs at least two samples and one component")
        pca = PCA(n_components=n_components, whiten=True, random_state=seed)
        Z = pca.fit_transform(Z)
    Z = normalize(Z)
    km = KMeans(n_clusters, n_init=20, random_state=seed).fit(Z)
    return km.labels_, pca, km


def train_deepcluster_pseudolabel(X, n_clusters=10, epochs=30, batch_size=256,
                                 lr=.05, seed=42, pca_dim=64, reassign=1,
                                 sobel=True, freeze_batchnorm_after=300,
                                 balance_clusters=False):
    side = math.isqrt(X.shape[1])
    if side * side != X.shape[1] or side < 8:
        raise ValueError("This CNN expects square images with side >= 8")
    if len(X) < 2 or batch_size < 2 or reassign < 1:
        raise ValueError("Use at least two images, batch_size >= 2 and reassign >= 1")
    seed_everything(seed)
    model = ConvNet(side, sobel).to(device)
    history = {'loss': []}
    batch_size = min(batch_size, len(X))
    for epoch in range(epochs):
        if epoch % reassign == 0:
            Z = vae_encode(X, model, batch_size)
            pseudo, _, _ = cluster_cnn_features(Z, n_clusters, seed, pca_dim)
            head = nn.Linear(128, n_clusters).to(device)
            nn.init.normal_(head.weight, std=.01)
            nn.init.zeros_(head.bias)
            parameters = list(model.parameters()) + list(head.parameters())
            optimizer = torch.optim.SGD(parameters, lr=lr, momentum=.9, weight_decay=1e-5)
            train_data = TensorDataset(X, torch.as_tensor(pseudo, dtype=torch.long))
            sampler = None
            if balance_clusters:
                weights = 1. / np.bincount(pseudo, minlength=n_clusters)[pseudo]
                sampler = WeightedRandomSampler(weights, len(X), replacement=True)
            # Dropping a short last batch also avoids singleton BatchNorm2d on 8x8 images.
            loader = DataLoader(train_data, batch_size=batch_size, sampler=sampler,
                                shuffle=sampler is None, drop_last=True)
        for group in optimizer.param_groups:
            group['lr'] = lr * .5 ** (epoch // 100)
        model.train()
        if freeze_batchnorm_after is not None and epoch >= freeze_batchnorm_after:
            for module in model.modules():
                if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
                    module.eval()
        total, count = 0., 0
        for x, target in loader:
            x, target = x.to(device), target.to(device)
            loss = F.cross_entropy(head(model(x)), target)
            gradient_step(loss, optimizer, parameters)
            total += len(x) * loss.item()
            count += len(x)
        history['loss'].append(total / count)
    Z = vae_encode(X, model, batch_size)
    pred, pca, km = cluster_cnn_features(Z, n_clusters, seed, pca_dim)
    return {'model': model, 'Z': Z, 'pred': pred, 'history': history,
            'pca': pca, 'clusterer': km}

## Joint AE / VAE + k-means
The early notebook selected the **farthest** center and detached the latent vector; its clustering weight was also always zero. Here the nearest-center loss keeps its gradient to the encoder, and the centers are updated. Set `cluster_weight=0` for the reconstruction-only comparison. For a VAE, clustering uses $\mu$, while reconstruction uses the sampled latent vector.

In [ ]:
class KMeansCriterion(nn.Module):
    def __init__(self, lmbda=1.):
        super().__init__()
        self.lmbda = lmbda

    def forward(self, embeddings, centroids):
        distances = (embeddings[:, None] - centroids[None]).square().sum(2)
        values, assignments = distances.min(1)
        return self.lmbda * values.mean(), assignments


def train_vae_kmeans(X, n_clusters=10, dimensions=None, pretrain_epochs=10,
                     epochs=30, batch_size=256, lr=1e-3, seed=42,
                     kind='vae', beta=1., cluster_weight=1., warm_epochs=10,
                     reconstruction='bce'):
    if kind not in ('ae', 'vae'):
        raise ValueError("kind must be ae or vae")
    seed_everything(seed)
    dimensions = dimensions or [X.shape[1], 500, 500, 2000, 10]
    network = VariationalAutoEncoder if kind == 'vae' else AutoEncoder
    model = network(dimensions, dropout=.2).to(device)
    pre_train(X, model, epochs=pretrain_epochs, batch_size=batch_size,
              lr=lr, beta=beta, loss=reconstruction, seed=seed)
    Z = vae_encode(X, model, batch_size)
    centers = KMeans(n_clusters, n_init=10, random_state=seed).fit(Z).cluster_centers_
    km = BatchKMeans(centers)
    criterion = KMeansCriterion(cluster_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': [], 'recon_loss': [], 'kl_loss': [], 'cluster_loss': []}
    for epoch in range(epochs):
        model.train()
        total = np.zeros(4)
        ramp = min(1., (epoch + 1) / max(1, warm_epochs))
        for x, _ in indexed_batches(X, batch_size):
            x = x.to(device)
            if kind == 'vae':
                recon, z, logvar = model(x)
                rec, kl = Loss_Function(recon, x, z, logvar, kind=reconstruction)
            else:
                recon, z = model(x)
                rec, _ = Loss_Function(recon, x, torch.zeros_like(z),
                                       torch.zeros_like(z), kind=reconstruction)
                kl = x.new_tensor(0.)
            cluster, labels = criterion(z, km.C)
            loss = rec + ramp * beta * kl + ramp * cluster
            gradient_step(loss, optimizer, model.parameters())
            km.update_centers(z.detach(), labels.detach())
            total += len(x) * np.array([loss.item(), rec.item(), kl.item(), cluster.item()])
        for name, value in zip(history, total / len(X)):
            history[name].append(value)
    Z = vae_encode(X, model, batch_size)
    pred = km.assign(torch.as_tensor(Z, device=device)).cpu().numpy()
    model.register_buffer('cluster_centers', km.C.clone())
    return {'model': model, 'Z': Z, 'pred': pred, 'history': history,
            'centers': km.C.cpu().numpy()}

## Run a comparison
Edit `methods` to choose experiments. Each starts from a new model. The default runs DEC and DCN on 2,000 MNIST images; add the others when needed. For the thesis-sized MLP, use `[X.shape[1], 500, 500, 2000, 10]` and increase both training budgets. Cross-method comparisons should record the budget and loss reductions, rather than assuming identical epoch counts make every method equally tuned.

In [ ]:
dataset = 'mnist'
n_samples = 2000
seed = 42
n_clusters = 10
pretrain_epochs = 3
epochs = 10
batch_size = 256
lr = 1e-3

methods = ['DEC', 'DCN']
# Other names: 'IDEC', 'VaDE', 'ADMM-kmeans', 'ADMM-gmm',
#              'DeepCluster', 'Joint-AE-kmeans', 'Joint-VAE-kmeans'
run_all_methods = False
seeds = [seed]  # e.g. [42, 43, 44] for repeated runs

X, y = load_data(dataset, n_samples=n_samples, seed=seed)
dimensions = [X.shape[1], 128, 64, 10]
print(X.shape, 'device:', device)

In [ ]:
results = {}
rows = []
all_methods = ['DEC', 'IDEC', 'DCN', 'VaDE', 'ADMM-kmeans', 'ADMM-gmm',
               'DeepCluster', 'Joint-AE-kmeans', 'Joint-VAE-kmeans']
if run_all_methods:
    methods = all_methods

for run_seed in seeds:
    arguments = dict(n_clusters=n_clusters, dimensions=dimensions,
                     pretrain_epochs=pretrain_epochs, epochs=epochs,
                     batch_size=batch_size, lr=lr, seed=run_seed)
    for name in methods:
        print(name, 'seed', run_seed)
        if name in ('DEC', 'IDEC'):
            result = train_dec(X, **arguments, idec=name == 'IDEC')
        elif name == 'DCN':
            result = train_dcn(X, **arguments)
        elif name == 'VaDE':
            result = train_vade(X, **arguments)
        elif name in ('ADMM-kmeans', 'ADMM-gmm'):
            result = train_dc_admm(X, **arguments, clustering=name.split('-')[1])
        elif name == 'DeepCluster':
            result = train_deepcluster_pseudolabel(X, n_clusters=n_clusters,
                        epochs=epochs, batch_size=batch_size, seed=run_seed)
        elif name in ('Joint-AE-kmeans', 'Joint-VAE-kmeans'):
            result = train_vae_kmeans(X, **arguments,
                                      kind='ae' if name == 'Joint-AE-kmeans' else 'vae')
        else:
            raise ValueError(f"Unknown method: {name}")
        results[(name, run_seed)] = result
        rows.append({'method': name, 'seed': run_seed, **score_dict(y, result['pred'])})

scores = pd.DataFrame(rows)
display(scores)
if len(seeds) > 1:
    display(scores.groupby('method')[['ACC', 'NMI', 'ARI', 'F1', 'Purity']].agg(['mean', 'std']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for (name, run_seed), result in results.items():
    values = result['history']['loss']
    ax.plot(np.arange(1, len(values) + 1), values, label=f'{name}, seed {run_seed}')
ax.set(xlabel='Epoch', ylabel='Training loss', title='Losses have different definitions')
ax.legend()
plt.tight_layout()
plt.show()

save_results = False
if save_results:
    Path('results').mkdir(exist_ok=True)
    scores.to_csv('results/deep_clustering_scores.csv', index=False)